# Eddy covariance spectral diagnostics

A reproducible example using synthetic data. Install `pip install -e ".[notebooks]"` from the repository first. No external data or downloads are required. All averaging periods below are seconds.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from TaylorSwift import ec_spectral as ecs

system = ecs.ECSystem(fs=20, avg_period=600, z_ref=3, lateral_sep=0.1)
u = 3.0
w, c = ecs.synthetic_turbulence(fs=system.fs, T=system.avg_period, seed=7)


## Align and estimate the cospectrum

Positive lag means the scalar trails the wind. Sensor attenuation may also shift the detected peak. The density is rescaled to the detrended covariance; use raw bins for integration. This differs from the existing `compute_cospectrum` normalization.


In [ ]:
lag, lags_s, r = ecs.find_lag(w, c, system.fs)
co = ecs.cospectrum(w, ecs.shift(c, lag), system.fs)
fit = ecs.fit_cospectrum(co["f_bin"], co["fCo_bin"], system.fs, system.avg_period)
np.testing.assert_allclose(co["Co"].sum() * system.fs / co["N"], co["cov"])
print(f"Lag: {lag / system.fs:.3f} s; covariance: {co['cov']:.3f}")
print(fit)


## Estimate losses and uncertainty

Integral and analytical factors use different approximations. The relative uncertainty is an analytical sensitivity estimate, not a fit confidence interval. Apply factors to covariances before WPL and do not apply a second spectral correction.


In [ ]:
taus = ecs.equivalent_time_constants(system, u)
F_int = ecs.correction_factor_integral(system, u, fx=fit["fx"], mu=fit["mu"])
F_an = ecs.correction_factor_analytical(fit["fx"], taus["tau_b"], taus["tau_e"])
dF = ecs.correction_uncertainty(fit["fx"], taus["tau_b"], taus["tau_e"])
eta_x = fit["fx"] * system.zd / u
print(f"Integral F={F_int:.3f}; analytical F={F_an:.3f}; relative uncertainty={dF:.1%}")


In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(17, 9))
ecs.plot_lag(lags_s, r, ax=axs[0, 0])
ecs.plot_cospectrum(co, fit, system, u, ax=axs[0, 1])
ecs.plot_ogive(co, ax=axs[0, 2])
ecs.plot_transfer_functions(system, u, ax=axs[1, 0])
ecs.plot_flux_loss(system, ax=axs[1, 1])
ecs.plot_correction_vs_wind(system, eta_x, ax=axs[1, 2])
fig.tight_layout()
plt.show()
plt.close(fig)


## Correct a table and compare closed-path response

Invalid wind speeds yield flagged NaN corrections. Large factors are flagged, not clipped. The reference peak frequency should be calibrated to suitable field intervals. Closed-path tube attenuation here does not include wall adsorption.


In [ ]:
table = pd.DataFrame({"u": [0.5, 1., 3., 5., 0.], "cov_wCO2": [-0.1, -0.2, -0.3, -0.2, -0.1]})
corrected = ecs.correct_flux_table(table, system, eta_x, cov_cols=("cov_wCO2",))
print(corrected.to_string(index=False))
closed = ecs.ECSystem(closed_path=True, flow_rate=6, tube_length=9, tau_scalar=0.1)
print(f"Tube Reynolds number: {closed.reynolds:.0f}; laminar: {closed.tube_laminar}")
fig, ax = plt.subplots()
ecs.plot_transfer_functions(closed, u, ax=ax)
fig.tight_layout()
plt.show()
plt.close(fig)
